# PanGBank Tutorial: Projecting AMR-Annotated Pangenomes onto a New Genome

This tutorial demonstrates how to:
1. Retrieve pangenomes from [PanGBank](https://pangbank.genoscope.cns.fr/)
2. Annotate gene families with [AMRFinderPlus](https://github.com/ncbi/amr)
3. Inject AMR metadata into the pangenome
4. Project the pangenome onto a query genome assembly
5. Explore projected AMR genes with interactive genome views

**Before starting, run the initialization cell below once.**

In [ ]:
%%capture pangbank_tutorials_init_logs

conda_command_path = "bin/micromamba"
amrfinder_env_path = "./amrfinder"
pip_command = "pip"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
from shutil import which
from pathlib import Path

conda_command = ""
if IN_COLAB:
    if not conda_command:
        conda_command_path = "bin/micromamba"
        !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj {conda_command_path}
        if not which(conda_command_path):
            raise RuntimeError("Micromamba installation failed")
        conda_command = conda_command_path
    else:
        if not which(conda_command_path):
            raise RuntimeError(f"conda_command_path: '{conda_command_path}' not found")
        conda_command = conda_command_path

    if not Path(amrfinder_env_path).exists():
        !{conda_command} create -y -p {amrfinder_env_path} -c conda-forge -c bioconda ncbi-amrfinderplus ppanggolin=2.3.0

    amrfinder_command = f"{conda_command} run -p {amrfinder_env_path} amrfinder"
    pp_command = f"{conda_command} run -p {amrfinder_env_path} ppanggolin"
    !{pip_command} install pangbank-cli pygenomeviz pyvis
else:
    amrfinder_command = f"amrfinder"
    pp_command = "ppanggolin"

!{amrfinder_command} -u

## Step 1: Download the Reference Pangenome

Retrieve the *Acinetobacter nosocomialis* pangenome from the GTDB_refseq collection using PanGBank.

In [ ]:
! pangbank search-pangenomes -t s__Acinetobacter_nosocomialis -c GTDB_refseq --download --release-version 2.0.0

## Step 1.1: Select the Downloaded Pangenome File

Set the pangenome file path used by all subsequent commands.

In [ ]:
pangenome_file = "pangbank/GTDB_refseq_s__Acinetobacter_nosocomialis_id10852.h5"

## Step 2: Extract Protein Family Sequences

Export all protein family sequences from the selected pangenome so they can be screened by AMRFinderPlus.

In [ ]:
! {pp_command} fasta -p {pangenome_file} -f --compress --prot_families all -o families_faa_output
families_protein_sequences = "families_faa_output/all_protein_families.faa.gz"

## Step 3: Annotate Protein Families with AMRFinderPlus

Run AMRFinderPlus on exported family proteins, format the output for PPanGGOLiN metadata ingestion, then attach AMR annotations to pangenome families.

In [ ]:
! {amrfinder_command} -p {families_protein_sequences} --plus --threads 8  -o amrfinder_result.tsv

In [ ]:
# Make the amrfinder output compatible with ppanggolin

# First column name should be 'families'
! sed -i '1s/\<Protein id\>/families/' amrfinder_result.tsv
# Remove space
! sed -i '1s/ /_/g' amrfinder_result.tsv
# Remove %
! sed -i 's/\%/Prct/g' amrfinder_result.tsv 

In [ ]:
# Add AMR annotations in the pangenome
! {pp_command} metadata --pangenome {pangenome_file} --metadata  amrfinder_result.tsv  --source amrfinder --assign families

## Step 4: Project the Pangenome onto a Query Genome

Download a query assembly and run PPanGGOLiN projection to map pangenome families and metadata onto this genome.

In [ ]:
! wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036/820/015/GCA_036820015.1_ASM3682001v1/GCA_036820015.1_ASM3682001v1_genomic.fna.gz  

In [ ]:
! {pp_command} projection -p {pangenome_file} --fasta GCA_036820015.1_ASM3682001v1_genomic.fna.gz --gff --proksee

---

# Data Analysis: Visualizing the Projected Genome

Render the Proksee/CGView JSON output to inspect projected genomic features in an interactive circular view.

In [ ]:
import json
from pathlib import Path

matches = list(Path(".").rglob("input_genome_proksee.json"))

with open(matches[0]) as ff:
    data = json.load(ff)

json_str = json.dumps(data)

html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/cgview/dist/cgview.css">
<script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
<script src="https://cdn.jsdelivr.net/npm/cgview/dist/cgview.min.js"></script>

<div id="cgview-container" style="width:800px; height:600px; border:1px solid #ccc;">
    Loading CGView...
</div>

<script>
(function() {{
    renderCGV();

    function renderCGV() {{
        const jsonData = {json_str};
        const width = 800;
        const height = 600;
    
        const container = document.getElementById("cgview-container");
        container.innerHTML = "";
    
        const viewer = new CGView.Viewer("cgview-container", {{
            width: width,
            height: height,
        }});
    
        window["cgview-container" + "_viewer"] = viewer;
        viewer.io.loadJSON(jsonData);
    }}
}})();
</script>
"""

In [ ]:
from IPython.display import HTML
HTML(html)

## Extract AMR Features from the Projected GFF

Load the projected GFF file, collect AMR-related attributes, and build a structured table for downstream exploration.

In [ ]:
from pygenomeviz.parser import Gff

AMR_FIELDS = [
    "family_amrfinder_Alignment_length",
    "family_amrfinder_Class",
    "family_amrfinder_Closest_reference_accession",
    "family_amrfinder_Closest_reference_name",
    "family_amrfinder_Element_name",
    "family_amrfinder_Element_symbol",
    "family_amrfinder_HMM_accession",
    "family_amrfinder_HMM_description",
    "family_amrfinder_Method",
    "family_amrfinder_Prct_Coverage_of_reference",
    "family_amrfinder_Prct_Identity_to_reference",
    "family_amrfinder_Reference_sequence_length",
    "family_amrfinder_Scope",
    "family_amrfinder_Subclass",
    "family_amrfinder_Subtype",
    "family_amrfinder_Target_length",
    "family_amrfinder_Type",
    "family_amrfinder_metadata_id"
]

matches = list(Path(".").rglob("input_genome.gff"))
gff = Gff(matches[0])

In [ ]:
amr_records = [record for record in gff.all_records if "family_amrfinder_Class" in record.attrs]
data = []
for record in amr_records:
    attrs = record.attrs
    data.append([
        record.seqid, record.type, record.start, record.end, record.strand,
        attrs["ID"][0], attrs["partition"][0], attrs["family"][0],
        *[attrs[e][0] if e in attrs else None for e in AMR_FIELDS]
    ])    

In [ ]:
import pandas as pd
df = pd.DataFrame(data, columns=["seqid", "type", "start", "end", "strand", "ID", "partition", "family"] + AMR_FIELDS)

## Inspect the Final AMR Annotation Table

Display the parsed AMR feature table for quick manual review.

In [ ]:
df